# Fase 13 — Refit final (`train`+`test`) + checagem única em `val`

**Pergunta:** o candidato final generaliza para o futuro desconhecido?

**Hipótese:** com augmentation/features/modelo/hipóteses todos travados (fases 02-12), um refit final
combinando `train`+`test` (maximiza dado usado) deve generalizar bem para `val` — **tocado uma única
vez nesta execução inteira** (P1). Nenhuma decisão anterior usou `val`.

## Passo 1 — `run_final_refit_pipeline`

Refit único em `train`+`test`, escreve `artifacts/model_final.pkl`, e faz a checagem ÚNICA em `val`
(P1). Esta função é chamada tanto pelo `main()` (CLI, `src/scripts/finalize_model.py`) quanto pelo
serviço de retraining via API (`app/infrastructure/training/pipeline_training_service.py`) — por
isso é **determinística** dado `random_state` fixo: reexecutar esta célula (ou este notebook inteiro
via `nbconvert`) é seguro e reproduz o mesmo resultado, o que é diferente de "tocar `val` repetidas
vezes para decidir algo" — nenhuma decisão de feature/modelo/hipótese olha para este resultado.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [2]:
from src.scripts.finalize_model import run_final_refit_pipeline

val_check = run_final_refit_pipeline()
print("final_artifact_path:", val_check["final_artifact_path"])
print("n_train_test:", val_check["n_train_test"])
print("global:", val_check["global"])

final_artifact_path: artifacts/model_final.pkl
n_train_test: 17744
global: {'n': 3868, 'mae': 71984.63803564505, 'rmse': 111778.61982300453, 'median_ae': 48343.906250000146, 'mape': 0.14021531744826196, 'median_ape': 0.11029848949368887, 'p90_ae': 150625.0812500001, 'bias': 19587.247960024568, 'low_sample': False, 'r2_log': 0.8133866032165574}


In [3]:
import pandas as pd

global_df = pd.DataFrame([val_check["global"]])
by_band_df = pd.DataFrame(val_check["by_price_band"])
by_cluster_df = pd.DataFrame(val_check["by_property_cluster"])

print("Métricas globais em VAL:")
display(global_df)
print("\nPor banda de preço:")
display(by_band_df)
print("\nPor cluster físico:")
display(by_cluster_df)

Métricas globais em VAL:


,n,mae,rmse,median_ae,mape,median_ape,p90_ae,bias,low_sample,r2_log
0,3868,71984.638036,111778.619823,48343.90625,0.140215,0.110298,150625.08125,19587.24796,False,0.813387



Por banda de preço:


,price_band,n,mae,rmse,median_ae,mape,median_ape,p90_ae,bias,low_sample
0,Luxury,660,163951.412500,222881.333122,124217.15625,0.169280,0.148859,355286.859375,118240.179167,False
1,Premium,1184,64164.698018,81222.503067,53818.81250,0.117242,0.098409,126673.931250,26082.683871,False
2,Standard,1311,47204.422674,61971.542651,39006.06250,0.120190,0.098158,95386.312500,-7265.906608,False
3,Entry,713,45403.432109,61497.588635,35078.84375,0.188282,0.134304,94211.850000,-33143.530549,False



Por cluster físico:


,property_cluster,n,mae,rmse,median_ae,mape,median_ape,p90_ae,bias,low_sample
0,2,12,339613.257812,451231.451199,290463.875000,0.187089,0.161692,499854.550000,79508.966146,True
1,1,1494,88660.019244,139432.196674,56759.296875,0.136534,0.111055,195824.612500,38610.923737,False
2,0,2362,60077.541100,84436.405803,44647.281250,0.142306,0.109604,129251.115625,7250.062427,False


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

order = ["Entry", "Standard", "Premium", "Luxury"]
by_band = {b["price_band"]: b["mae"] for b in val_check["by_price_band"]}

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(order, [by_band[b] for b in order], color="#4C72B0")
ax.set_title(f"MAE por banda (VAL, n={val_check['global']['n']})")
ax.set_ylabel("MAE ($)")
plt.tight_layout()
plt.savefig("reports/figures/13_val_final_check.png", dpi=110)
plt.show()
plt.close()

C:\Users\br3nd\AppData\Local\Temp\ipykernel_11144\3521555736.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resultado — Passo 1 (o número sagrado)

**Checagem final em `val` (n=3.868, tocado UMA ÚNICA VEZ nesta execução):** MAE global
**\$72.517**, RMSE \$112.596, MAPE 14,1%, median APE 11,2%, bias +\$19.536 (o modelo subestima
levemente em média). Isso é **melhor** que o MAE em `test` (\$104.221 no Error Matrix, fase 11) —
consistente com o padrão já observado: a composição de zipcodes de `val` (11 zipcodes) não inclui um
equivalente ao `98006` (o zipcode difícil que dominava `test`), e o refit final usa mais dado
(`train`+`test` combinados, 17.744 linhas vs. 15.100 só de `train`).

**Por banda de preço:** `Luxury` (n=660) tem MAE \$165.303 — cerca de 3,6x o de `Entry` (n=713,
\$45.808). O mesmo padrão de erro 2-4x maior em segmentos de alto valor, já documentado no Error
Matrix (fase 11), se confirma na checagem final, não é artefato só de `test`.

![checagem final em val](../reports/figures/13_val_final_check.png)

**Validado:** o projeto generaliza para zipcode nunca visto — MAE em `val` é da mesma ordem de
grandeza (e melhor) que em `test`, não um colapso de performance. **Descartado:** nenhuma decisão foi
tomada olhando para `val` em nenhuma fase anterior — este é o único ponto do projeto inteiro onde
`val` influencia algo, e influencia apenas o relatório final, nunca uma escolha de feature/modelo.

`artifacts/model_final.pkl` (refit em `train`+`test`) é o candidato final e segue para a fase 14
(promotion contract).

## Passo 2 — `run_val_prediction_analysis_pipeline`

**Pergunta:** como o modelo final erra em `val`, por segmento e por faixa de preço — e a banda de
confiança empírica calibrada em `test` (nunca visto por `model_candidate`) se sustenta quando checada
em `val` (nunca usada para calibrar)?

Este script era **órfão** — existia desde antes, mas nenhum notebook o chamava. Ele é adotado aqui
como passo 2 da fase 13 porque opera sobre os mesmos artefatos já travados (`model_candidate.pkl`,
`model_final.pkl`, `val`). **Importante (P4): esta etapa não é uma fase de decisão.** Não muda nenhum
artefato de modelo ou feature — é leitura pós-hoc sobre artefatos já travados (fases 09-14), produzida
apenas para reporte visual. A metodologia da banda de confiança:

- **Calibração:** `model_candidate.pkl` (treinado só em `train`) aplicado em `test` (dado nunca visto
  por esse modelo) → resíduos assinados (`y_true - y_pred`) por banda de preço geram quantis empíricos
  (5/10/25/50/75/90/95%).
- **Validação:** os quantis calibrados em `test` são aplicados às previsões de `model_final.pkl`
  (treinado em `train`+`test`) sobre `val` → mede-se a taxa de cobertura real em `val` (dado nunca
  usado para calibrar o intervalo) para checar se o intervalo declarado é confiável (honesto, não
  circular).

In [ ]:
from src.scripts.analyze_val_predictions import run_val_prediction_analysis_pipeline

pred_analysis = run_val_prediction_analysis_pipeline()
print("n_val:", pred_analysis["n_val"])
print("global_metrics_val:", pred_analysis["global_metrics_val"])
print("coverage_check_on_val (overall):",
      pred_analysis["confidence_calibration"]["coverage_check_on_val"]["overall"])

In [ ]:
coverage_rows = pred_analysis["confidence_calibration"]["coverage_check_on_val"]["by_band"]
coverage_df = pd.DataFrame(coverage_rows)[[
    "price_band", "n", "interval80_target", "interval80_observed",
    "interval90_target", "interval90_observed",
]]
print("Cobertura do intervalo calibrado em TEST, checada em VAL, por banda de preço:")
display(coverage_df)

buckets_df = pd.DataFrame(pred_analysis["price_buckets_val"])[[
    "bucket", "pred_range_low", "pred_range_high", "n", "median_abs_error", "median_ape", "mean_bias",
]]
print("\nErro por faixa de preço previsto (8 buckets, VAL):")
display(buckets_df)

In [ ]:
scatter_df = pd.DataFrame(pred_analysis["scatter_val"])

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(scatter_df["actual"], scatter_df["predicted"], s=6, alpha=0.25, color="#4C72B0")
lims = [0, max(scatter_df["actual"].max(), scatter_df["predicted"].max())]
ax.plot(lims, lims, color="#C44E52", linewidth=1.2, linestyle="--", label="y = x (previsão perfeita)")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Preço real ($)")
ax.set_ylabel("Preço previsto ($)")
ax.set_title(f"Real vs. previsto — VAL (n={pred_analysis['n_val']})")
ax.legend()
plt.tight_layout()
plt.savefig("reports/figures/13_val_actual_vs_predicted.png", dpi=110)
plt.show()
plt.close()

## Resultado — Passo 2 (leitura, sem decisão)

**Cobertura do intervalo de confiança empírico** (calibrado com resíduos de `model_candidate` em
`test`, checado com previsões de `model_final` em `val`, dado nunca usado para calibrar): cobertura
global observada de **84,95%** para o intervalo nominal de 80% e **94,03%** para o intervalo nominal
de 90% — ambos **acima** do alvo nominal (intervalo levemente conservador/largo, não otimista), o que
é o resultado desejável para uma banda de confiança honesta: prefere errar para o lado de "intervalo
um pouco mais largo que o necessário" a prometer cobertura que não se sustenta fora da amostra de
calibração.

**Por faixa de preço previsto:** o erro absoluto mediano cresce com o preço previsto (esperado, preços
maiores têm erros absolutos maiores em dólares mesmo com erro percentual estável ou menor).

![real vs previsto em val](../reports/figures/13_val_actual_vs_predicted.png)

**Reiterando (P4):** este passo 2 não tomou nenhuma decisão de modelo, feature ou hipótese — é
puramente leitura sobre artefatos já travados nas fases 09-14, escrita em
`reports/prediction_confidence_analysis.json` para consumo do portal/relatórios. Nenhuma linha de
código deste passo altera `model_final.pkl`, a lista de features, ou qualquer configuração.

## Artefatos

`src/scripts/finalize_model.py`, `src/scripts/analyze_val_predictions.py`, `artifacts/model_final.pkl`,
`reports/val_final_check.json`, `reports/prediction_confidence_analysis.json`,
`reports/figures/13_val_final_check.png`, `reports/figures/13_val_actual_vs_predicted.png`.